# Demo: Model Calibration Concept

## Objectives

This notebook demonstrates the fundamental concept of model calibration using a simple synthetic example. We explore:

- How a model parameter controls the simulated system response
- How we quantify the mismatch between model and observations (objective function)
- How gradient-based optimization finds the best parameter value
- Why calibration works and what its limitations are

## Background

Calibration is the process of adjusting model parameters so that simulated outputs match observed measurements. In groundwater modeling, we typically have:

- **Parameters**: Properties we don't know exactly (e.g., hydraulic conductivity, recharge rate)
- **Observations**: Measurements from the real system (e.g., groundwater levels, spring discharge)
- **Objective function**: A metric quantifying how well the model matches observations

The calibration algorithm searches through parameter space to minimize the objective function.

> **Theory: Objective Function**
> 
> The most common objective function is the Sum of Squared Residuals (SSR):
> 
> SSR = Σ(h_sim - h_obs)²
> 
> Where:
> - h_sim = simulated head at observation location
> - h_obs = observed head
> - The sum is over all observations
>
> The goal of calibration is to find parameter values that minimize SSR.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets

# Import common setup - same model and observations as all calibration demos
from calibration_common import (
    X_DOMAIN, L, H1, H2, K_TRUE, N_TRUE,
    X_OBS_STANDARD, H_OBS_STANDARD, H_TRUE_STANDARD,
    gw_model_1d, objective_function, NOISE_STD
)

# Set up plotting
%matplotlib widget
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. The Synthetic Model

We use a simple "model" that represents an unconfined aquifer with steady-state flow (Dupuit assumption). The model has:

- **One parameter to calibrate**: Hydraulic conductivity (K)
- **Physics**: Darcy's law with the Dupuit assumption determines the head distribution
- **Boundary conditions**: Fixed heads at both ends

The Dupuit equation for steady-state 1D unconfined flow with recharge is:

$$h^2(x) = h_1^2 - (h_1^2 - h_2^2)\frac{x}{L} + \frac{N}{K} x (L - x)$$

Where:
- $h_1$ = head at left boundary (x=0)
- $h_2$ = head at right boundary (x=L)
- $N$ = recharge rate
- $K$ = hydraulic conductivity
- $L$ = domain length

This means K directly controls the "bulge" in the head profile!

In [ ]:
# Display the standard observations we're working with
x_obs = X_OBS_STANDARD
h_obs = H_OBS_STANDARD

print("Synthetic observation data:")
print("=" * 50)
print(f"{'Well':^8} {'X (m)':^10} {'h_obs (m)':^12} {'h_true (m)':^12}")
print("-" * 50)
for i, (x, ho, ht) in enumerate(zip(x_obs, h_obs, H_TRUE_STANDARD)):
    print(f"{i+1:^8} {x:^10.0f} {ho:^12.2f} {ht:^12.2f}")
print("-" * 50)
print(f"Measurement noise std: {NOISE_STD} m")
print(f"\nDomain: L = {L} m, h₁ = {H1} m, h₂ = {H2} m")
print(f"Recharge: N = {N_TRUE} m/d = {N_TRUE*1000:.1f} mm/d")

## 2. The Objective Function

The objective function measures how well a given parameter value reproduces the observations. We use the Sum of Squared Residuals (SSR).

In [ ]:
def objective_gradient(K, delta_K=0.01):
    """
    Estimate gradient of objective function using finite differences.
    
    Args:
        K: Current hydraulic conductivity (m/day)
        delta_K: Step size for finite difference
    
    Returns:
        gradient: dSSR/dK
    """
    SSR_plus = objective_function(K + delta_K)
    SSR_minus = objective_function(K - delta_K)
    gradient = (SSR_plus - SSR_minus) / (2 * delta_K)
    return gradient

### Visualizing the Objective Function Surface

Let's see how the objective function varies with K. The minimum indicates the best-fit parameter value.

In [ ]:
# Calculate objective function over a range of K values
K_range = np.linspace(5, 50, 100)
SSR_values = [objective_function(K) for K in K_range]

# Find minimum
K_min_idx = np.argmin(SSR_values)
K_optimal = K_range[K_min_idx]

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(K_range, SSR_values, 'b-', linewidth=2, label='Objective function')
ax.axvline(K_TRUE, color='green', linestyle='--', linewidth=1.5, label=f'True K = {K_TRUE} m/day')
ax.axvline(K_optimal, color='red', linestyle=':', linewidth=1.5, label=f'Optimal K ≈ {K_optimal:.1f} m/day')
ax.scatter([K_optimal], [SSR_values[K_min_idx]], color='red', s=100, zorder=5)

ax.set_xlabel('Hydraulic Conductivity K (m/day)')
ax.set_ylabel('Sum of Squared Residuals (m²)')
ax.set_title('Objective Function Surface')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(5, 50)

plt.tight_layout()
plt.show()

print(f"\nThe minimum SSR occurs at K ≈ {K_optimal:.2f} m/day")
print(f"True K = {K_TRUE} m/day")
print(f"Difference due to measurement noise: {abs(K_optimal - K_TRUE):.2f} m/day")

> **Note: Effect of Measurement Noise**
> 
> The optimal K from calibration doesn't exactly match the "true" K due to measurement noise.
> This illustrates an important concept: **calibration finds the best fit to noisy data, not the true parameter.**
> 
> In real applications, we never know the "true" parameter - we only have our observations!

## 3. Gradient Descent Optimization

Now let's implement a simple gradient descent optimizer. The algorithm:

1. Start with an initial guess for K
2. Calculate the gradient (slope) of the objective function
3. Move in the direction of steepest descent
4. Repeat until convergence

The update rule is:

K_new = K_old - α × (dSSR/dK)

Where α is the learning rate (step size).

In [ ]:
def gradient_descent(K_init, learning_rate=0.5, max_iter=50, tol=1e-6):
    """
    Perform gradient descent to minimize the objective function.
    
    Args:
        K_init: Initial guess for K (m/day)
        learning_rate: Step size multiplier
        max_iter: Maximum number of iterations
        tol: Convergence tolerance
    
    Returns:
        history: Dictionary with optimization history
    """
    K = K_init
    history = {
        'K': [K],
        'SSR': [objective_function(K)],
        'gradient': []
    }
    
    for i in range(max_iter):
        # Calculate gradient
        grad = objective_gradient(K)
        history['gradient'].append(grad)
        
        # Update K (move in direction of steepest descent)
        K_new = K - learning_rate * grad
        
        # Ensure K stays positive
        K_new = max(K_new, 0.1)
        
        # Store history
        SSR_new = objective_function(K_new)
        history['K'].append(K_new)
        history['SSR'].append(SSR_new)
        
        # Check convergence
        if abs(K_new - K) < tol:
            print(f"Converged after {i+1} iterations")
            break
        
        K = K_new
    
    return history

### Run the Calibration

Let's start with a poor initial guess and watch the optimizer find the optimal K.

In [ ]:
# Run gradient descent from a poor initial guess
K_initial = 40.0  # Start far from the true value
history = gradient_descent(K_initial, learning_rate=0.3, max_iter=30)

print(f"\nCalibration Results:")
print(f"===================")
print(f"Initial K: {K_initial:.2f} m/day")
print(f"Final K:   {history['K'][-1]:.2f} m/day")
print(f"True K:    {K_TRUE:.2f} m/day")
print(f"\nInitial SSR: {history['SSR'][0]:.4f} m²")
print(f"Final SSR:   {history['SSR'][-1]:.4f} m²")
print(f"Reduction:   {(1 - history['SSR'][-1]/history['SSR'][0])*100:.1f}%")

## 4. Visualizing the Calibration Process

The following visualization shows:
- **Left panel**: The head profile evolving as K changes
- **Right panel**: The optimizer's path on the objective function surface

In [ ]:
# Create figure with two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: Head profiles
ax1.set_xlabel('Distance (m)')
ax1.set_ylabel('Hydraulic Head (m)')
ax1.set_title('Model Response vs Observations')
ax1.set_xlim(0, L)
ax1.set_ylim(94, 102)
ax1.grid(True, alpha=0.3)

# Plot observations (fixed)
ax1.scatter(x_obs, h_obs, color='red', s=80, zorder=5, label='Observations', marker='o')

# Initialize model line (will be updated)
line_model, = ax1.plot([], [], 'b-', linewidth=2, label='Model')
ax1.legend(loc='upper right')

# Add iteration text
iter_text = ax1.text(0.02, 0.98, '', transform=ax1.transAxes, fontsize=11,
                     verticalalignment='top', fontfamily='monospace',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Right panel: Objective function with optimizer path
ax2.plot(K_range, SSR_values, 'b-', linewidth=2, alpha=0.7)
ax2.axvline(K_TRUE, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label=f'True K')
ax2.set_xlabel('Hydraulic Conductivity K (m/day)')
ax2.set_ylabel('Sum of Squared Residuals (m²)')
ax2.set_title('Optimization Path')
ax2.set_xlim(5, 50)
ax2.grid(True, alpha=0.3)

# Initialize path markers
path_line, = ax2.plot([], [], 'ro-', markersize=6, linewidth=1.5, alpha=0.7)
current_point, = ax2.plot([], [], 'ro', markersize=12)
ax2.legend(loc='upper right')

plt.tight_layout()

def update(frame):
    """Update function for animation."""
    # Get current K and SSR
    K_current = history['K'][frame]
    SSR_current = history['SSR'][frame]
    
    # Update model line
    x, h = gw_model_1d(K_current)
    line_model.set_data(x, h)
    
    # Update text
    iter_text.set_text(f'Iteration: {frame}\nK = {K_current:.2f} m/day\nSSR = {SSR_current:.4f} m²')
    
    # Update optimization path
    path_K = history['K'][:frame+1]
    path_SSR = history['SSR'][:frame+1]
    path_line.set_data(path_K, path_SSR)
    current_point.set_data([K_current], [SSR_current])
    
    return line_model, iter_text, path_line, current_point

# Create animation
n_frames = len(history['K'])
ani = FuncAnimation(fig, update, frames=n_frames, interval=500, blit=True, repeat=True)

plt.show()

> **Key Observations**
> 
> Watch how:
> 1. The optimizer starts with a poor K value (model far from observations)
> 2. Each iteration moves K in the direction of steepest descent
> 3. The model progressively gets closer to the observations
> 4. Eventually the optimizer converges near the minimum

## 5. Interactive Exploration

Use the slider below to manually explore how changing K affects the model fit.

In [ ]:
# Create interactive figure
fig_int, (ax_model, ax_obj) = plt.subplots(1, 2, figsize=(12, 5))

# Model panel
ax_model.scatter(x_obs, h_obs, color='red', s=80, zorder=5, label='Observations', marker='o')
model_line_int, = ax_model.plot([], [], 'b-', linewidth=2, label='Model')
ax_model.set_xlabel('Distance (m)')
ax_model.set_ylabel('Hydraulic Head (m)')
ax_model.set_title('Model vs Observations')
ax_model.set_xlim(0, L)
ax_model.set_ylim(94, 102)
ax_model.grid(True, alpha=0.3)
ax_model.legend(loc='upper right')

# Objective function panel
ax_obj.plot(K_range, SSR_values, 'b-', linewidth=2)
ax_obj.axvline(K_TRUE, color='green', linestyle='--', linewidth=1.5, label=f'True K = {K_TRUE}')
current_marker, = ax_obj.plot([], [], 'ro', markersize=15)
ax_obj.set_xlabel('Hydraulic Conductivity K (m/day)')
ax_obj.set_ylabel('Sum of Squared Residuals (m²)')
ax_obj.set_title('Objective Function')
ax_obj.set_xlim(5, 50)
ax_obj.grid(True, alpha=0.3)
ax_obj.legend(loc='upper right')

# Info text
info_text = ax_model.text(0.02, 0.98, '', transform=ax_model.transAxes, fontsize=11,
                          verticalalignment='top', fontfamily='monospace',
                          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()

def update_interactive(K_value):
    """Update plots based on slider value."""
    # Calculate model
    x, h = gw_model_1d(K_value)
    SSR = objective_function(K_value)
    
    # Update model line
    model_line_int.set_data(x, h)
    
    # Update objective function marker
    current_marker.set_data([K_value], [SSR])
    
    # Update info text
    info_text.set_text(f'K = {K_value:.1f} m/day\nSSR = {SSR:.4f} m²')
    
    fig_int.canvas.draw_idle()

# Create slider widget
K_slider = widgets.FloatSlider(
    value=25.0,
    min=5.0,
    max=50.0,
    step=0.5,
    description='K (m/day):',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

# Link slider to update function
widgets.interactive(update_interactive, K_value=K_slider)

# Initialize plot
update_interactive(25.0)

display(K_slider)
plt.show()

## 6. Comparing Starting Points

Let's see how the optimization behaves from different starting points.

In [ ]:
# Run optimization from different starting points
starting_points = [8, 20, 35, 45]
colors = ['purple', 'orange', 'teal', 'brown']

fig, ax = plt.subplots(figsize=(10, 6))

# Plot objective function
ax.plot(K_range, SSR_values, 'b-', linewidth=2, alpha=0.5, label='Objective function')
ax.axvline(K_TRUE, color='green', linestyle='--', linewidth=2, label=f'True K = {K_TRUE}')

# Run and plot from each starting point
for K_start, color in zip(starting_points, colors):
    hist = gradient_descent(K_start, learning_rate=0.3, max_iter=25)
    
    # Plot path
    ax.plot(hist['K'], hist['SSR'], 'o-', color=color, markersize=6, 
            linewidth=1.5, alpha=0.8, label=f'Start: K={K_start}')
    
    # Mark start and end
    ax.scatter([hist['K'][0]], [hist['SSR'][0]], color=color, s=150, marker='s', zorder=5)
    ax.scatter([hist['K'][-1]], [hist['SSR'][-1]], color=color, s=150, marker='*', zorder=5)

ax.set_xlabel('Hydraulic Conductivity K (m/day)', fontsize=12)
ax.set_ylabel('Sum of Squared Residuals (m²)', fontsize=12)
ax.set_title('Convergence from Different Starting Points', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(5, 50)

plt.tight_layout()
plt.show()

print("\nAll paths converge to approximately the same minimum!")
print("(For this simple problem with a single global minimum)")

## 7. Key Takeaways

> **Summary: Model Calibration**
> 
> 1. **Calibration finds parameters** that minimize the mismatch between model and observations
> 
> 2. **Gradient descent** follows the steepest descent direction to find the minimum
> 
> 3. **Measurement noise** causes the calibrated parameter to differ slightly from the "true" value
> 
> 4. **Convergence** depends on the learning rate and the shape of the objective function
> 
> 5. **Single-minimum problems** (like this one) are well-behaved - gradient descent will find the solution

> **Real-World Complications**
> 
> This simple example has one parameter and one clear minimum. Real groundwater models have:
> 
> - **Multiple parameters**: The objective function becomes a surface in high-dimensional space
> - **Parameter correlations**: Different parameter combinations can produce similar fits
> - **Local minima**: The optimizer might get stuck in suboptimal solutions
> - **Non-uniqueness**: Multiple parameter sets may fit the data equally well
> 
> Advanced techniques like PEST, UCODE, or ensemble methods address these challenges.

## References

- Doherty, J. (2015). Calibration and Uncertainty Analysis for Complex Environmental Models. Watermark Numerical Computing.
- Hill, M.C. & Tiedeman, C.R. (2007). Effective Groundwater Model Calibration. Wiley.
- Anderson, M.P., Woessner, W.W., & Hunt, R.J. (2015). Applied Groundwater Modeling (2nd ed.). Academic Press.